In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ml-latest-small"

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")

# Chronological ordering within each user
ratings = ratings.sort_values(
    ["userId", "timestamp", "movieId"]
).reset_index(drop=True)

# Reserve the latest 20% of ratings per user for testing.
# Keep at least 5 training ratings and 1 test rating per user.
train_parts = []
test_parts = []

for user_id, user_ratings in ratings.groupby("userId", sort=False):
    n = len(user_ratings)
    n_test = max(1, int(np.ceil(n * 0.2)))
    n_test = min(n_test, n - 5)

    train_parts.append(user_ratings.iloc[:-n_test])
    test_parts.append(user_ratings.iloc[-n_test:])

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)

print("Training ratings:", len(train))
print("Test ratings:", len(test))
print("Training users:", train["userId"].nunique())
print("Test users:", test["userId"].nunique())
print("Training movies:", train["movieId"].nunique())
print("Test ratings per user — summary:")
display(test.groupby("userId").size().describe())

assert set(train["userId"]) == set(test["userId"])
assert not set(zip(train["userId"], train["movieId"])) & set(
    zip(test["userId"], test["movieId"])
)

print("Split checks passed.")

Training ratings: 80419
Test ratings: 20417
Training users: 610
Test users: 610
Training movies: 8230
Test ratings per user — summary:


count    610.000000
mean      33.470492
std       53.887250
min        4.000000
25%        7.000000
50%       14.500000
75%       34.000000
max      540.000000
dtype: float64

Split checks passed.


In [2]:
# Movies each user has already rated in training
seen_by_user = (
    train.groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

# Rank movies by training-set rating count
movie_popularity = (
    train.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        average_rating=("rating", "mean")
    )
    .reset_index()
    .merge(movies, on="movieId")
    .sort_values(
        ["rating_count", "average_rating"],
        ascending=False
    )
)

# Only movies present in the training data can be recommended
available_movies = movie_popularity["movieId"].tolist()

def recommend_popular(user_id, k=10):
    seen = seen_by_user.get(user_id, set())

    recommendations = [
        movie_id
        for movie_id in available_movies
        if movie_id not in seen
    ]

    return recommendations[:k]

# Preview recommendations for one user
example_user = 1
example_recommendations = recommend_popular(example_user, k=10)

display(
    movies[movies["movieId"].isin(example_recommendations)]
    .set_index("movieId")
    .loc[example_recommendations]
    .reset_index()[["movieId", "title", "genres"]]
)

,movieId,title,genres
0,318,"Shawshank Redemption, The (1994)",Crime|Drama
1,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
2,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
3,2959,Fight Club (1999),Action|Crime|Drama|Thriller
4,150,Apollo 13 (1995),Adventure|Drama|IMAX
5,527,Schindler's List (1993),Drama|War
6,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
7,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller
8,858,"Godfather, The (1972)",Crime|Drama
9,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy


In [3]:
def ranking_metrics(recommendations, relevant_items, k=10):
    recommendations = recommendations[:k]
    relevant_items = set(relevant_items)

    if not relevant_items:
        return None

    hits = [
        1 if item in relevant_items else 0
        for item in recommendations
    ]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_items)

    discounts = np.log2(np.arange(2, len(hits) + 2))
    dcg = sum(hit / discount for hit, discount in zip(hits, discounts))

    ideal_hits = min(len(relevant_items), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return precision, recall, ndcg


# Relevant held-out movies: rating >= 4.0
# Exclude movies unavailable in the training catalog.
train_movie_ids = set(train["movieId"])
test_relevant = (
    test[test["rating"] >= 4.0]
    .loc[lambda df: df["movieId"].isin(train_movie_ids)]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

results = []

for user_id in test["userId"].unique():
    relevant = test_relevant.get(user_id, set())

    if not relevant:
        continue

    recommendations = recommend_popular(user_id, k=10)
    metrics = ranking_metrics(recommendations, relevant, k=10)

    if metrics is not None:
        results.append(metrics)

baseline_results = pd.DataFrame(
    results,
    columns=["Precision@10", "Recall@10", "NDCG@10"]
)

print("Users evaluated:", len(baseline_results))
display(baseline_results.mean().to_frame("Popularity Baseline"))

Users evaluated: 591


,Popularity Baseline
Precision@10,0.058037
Recall@10,0.049802
NDCG@10,0.075741


In [4]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Build a user-movie matrix from training data only
user_ids = sorted(train["userId"].unique())
movie_ids = sorted(train["movieId"].unique())

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

rows = train["userId"].map(user_to_idx).to_numpy()
cols = train["movieId"].map(movie_to_idx).to_numpy()
values = train["rating"].to_numpy(dtype=float)

user_movie_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)

# Similarity between movies based on rating patterns
movie_similarity = cosine_similarity(
    user_movie_matrix.T,
    dense_output=True
)

print("User-movie matrix:", user_movie_matrix.shape)
print("Movie similarity matrix:", movie_similarity.shape)
print("Similarity matrix created.")

User-movie matrix: (610, 8230)
Movie similarity matrix: (8230, 8230)
Similarity matrix created.


In [5]:
def recommend_item_based(user_id, k=10, min_ratings=3):
    if user_id not in user_to_idx:
        return recommend_popular(user_id, k)

    user_idx = user_to_idx[user_id]
    user_ratings = user_movie_matrix.getrow(user_idx)

    seen_indices = user_ratings.indices
    seen_movie_ids = {movie_ids[i] for i in seen_indices}

    if len(seen_indices) == 0:
        return recommend_popular(user_id, k)

    # Weighted sum of the user's ratings by similarity to each candidate
    scores = movie_similarity[:, seen_indices] @ user_ratings.data
    scores = np.asarray(scores).ravel()

    # Do not recommend movies already rated by this user
    scores[seen_indices] = -np.inf

    # Require candidates to have at least min_ratings in training
    rating_counts = np.asarray((user_movie_matrix > 0).sum(axis=0)).ravel()
    scores[rating_counts < min_ratings] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]
    return [
        movie_ids[i]
        for i in top_indices
        if np.isfinite(scores[i])
    ]

# Preview recommendations
example_recommendations = recommend_item_based(1, k=10)

display(
    movies[movies["movieId"].isin(example_recommendations)]
    .set_index("movieId")
    .loc[example_recommendations]
    .reset_index()[["movieId", "title", "genres"]]
)

,movieId,title,genres
0,1240,"Terminator, The (1984)",Action|Sci-Fi|Thriller
1,1265,Groundhog Day (1993),Comedy|Fantasy|Romance
2,2918,Ferris Bueller's Day Off (1986),Comedy
3,2716,Ghostbusters (a.k.a. Ghost Busters) (1984),Action|Comedy|Sci-Fi
4,1036,Die Hard (1988),Action|Crime|Thriller
5,1391,Mars Attacks! (1996),Action|Comedy|Sci-Fi
6,3033,Spaceballs (1987),Comedy|Sci-Fi
7,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi
8,1387,Jaws (1975),Action|Horror
9,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi


In [6]:
cf_results = []

for user_id in test["userId"].unique():
    relevant = test_relevant.get(user_id, set())

    if not relevant:
        continue

    recommendations = recommend_item_based(user_id, k=10)
    metrics = ranking_metrics(recommendations, relevant, k=10)

    if metrics is not None:
        cf_results.append(metrics)

cf_results = pd.DataFrame(
    cf_results,
    columns=["Precision@10", "Recall@10", "NDCG@10"]
)

comparison = pd.DataFrame({
    "Popularity Baseline": baseline_results.mean(),
    "Item-Based Collaborative Filtering": cf_results.mean()
})

display(comparison)

print("Users evaluated — popularity:", len(baseline_results))
print("Users evaluated — collaborative filtering:", len(cf_results))

,Popularity Baseline,Item-Based Collaborative Filtering
Precision@10,0.058037,0.072758
Recall@10,0.049802,0.073998
NDCG@10,0.075741,0.090617


Users evaluated — popularity: 591
Users evaluated — collaborative filtering: 591


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Subtract each user's mean training rating
user_means = train.groupby("userId")["rating"].mean()

centered_train = train.copy()
centered_train["centered_rating"] = (
    centered_train["rating"]
    - centered_train["userId"].map(user_means)
)

rows = centered_train["userId"].map(user_to_idx).to_numpy()
cols = centered_train["movieId"].map(movie_to_idx).to_numpy()
values = centered_train["centered_rating"].to_numpy(dtype=float)

centered_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)

centered_similarity = cosine_similarity(
    centered_matrix.T,
    dense_output=True
)

print("Mean-centered similarity matrix:", centered_similarity.shape)

Mean-centered similarity matrix: (8230, 8230)


In [8]:
def recommend_centered(user_id, k=10, min_ratings=3):
    if user_id not in user_to_idx:
        return recommend_popular(user_id, k)

    user_idx = user_to_idx[user_id]
    rated_indices = user_movie_matrix.getrow(user_idx).indices

    if len(rated_indices) == 0:
        return recommend_popular(user_id, k)

    # Use centered ratings for scoring
    centered_ratings = centered_matrix.getrow(user_idx)
    scores = (
        centered_similarity[:, centered_ratings.indices]
        @ centered_ratings.data
    )
    scores = np.asarray(scores).ravel()

    scores[rated_indices] = -np.inf

    rating_counts = np.asarray(
        (user_movie_matrix > 0).sum(axis=0)
    ).ravel()
    scores[rating_counts < min_ratings] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]

    return [
        movie_ids[i]
        for i in top_indices
        if np.isfinite(scores[i])
    ]

In [9]:
centered_results = []

for user_id in test["userId"].unique():
    relevant = test_relevant.get(user_id, set())

    if not relevant:
        continue

    recommendations = recommend_centered(user_id, k=10)
    metrics = ranking_metrics(recommendations, relevant, k=10)

    if metrics is not None:
        centered_results.append(metrics)

centered_results = pd.DataFrame(
    centered_results,
    columns=["Precision@10", "Recall@10", "NDCG@10"]
)

comparison = pd.DataFrame({
    "Popularity Baseline": baseline_results.mean(),
    "Item-Based CF": cf_results.mean(),
    "Mean-Centered CF": centered_results.mean()
})

display(comparison)

,Popularity Baseline,Item-Based CF,Mean-Centered CF
Precision@10,0.058037,0.072758,0.041117
Recall@10,0.049802,0.073998,0.039716
NDCG@10,0.075741,0.090617,0.052121


In [10]:
# Define the catalog available to recommenders
train_movie_ids = set(train["movieId"])

# Relevant test items must:
# 1. Have a held-out rating >= 4.0
# 2. Exist in the training catalog
# 3. Not have been rated by that user during training
test_relevant = {}

for user_id, user_test in test.groupby("userId"):
    seen = seen_by_user.get(user_id, set())

    relevant = set(
        user_test.loc[
            (user_test["rating"] >= 4.0)
            & (user_test["movieId"].isin(train_movie_ids)),
            "movieId"
        ]
    )

    relevant -= seen

    if relevant:
        test_relevant[user_id] = relevant

print("Users with at least one relevant test movie:", len(test_relevant))
print(
    "Relevant held-out movies:",
    sum(len(items) for items in test_relevant.values())
)

Users with at least one relevant test movie: 591
Relevant held-out movies: 8882


In [11]:
def evaluate_recommender(recommender, test_relevant, k=10):
    results = []

    for user_id, relevant in test_relevant.items():
        recommendations = recommender(user_id, k=k)
        metrics = ranking_metrics(recommendations, relevant, k=k)

        if metrics is not None:
            results.append(metrics)

    return pd.DataFrame(
        results,
        columns=["Precision@10", "Recall@10", "NDCG@10"]
    )


baseline_final = evaluate_recommender(
    recommend_popular, test_relevant
)

cf_final = evaluate_recommender(
    recommend_item_based, test_relevant
)

final_comparison = pd.DataFrame({
    "Popularity Baseline": baseline_final.mean(),
    "Item-Based Collaborative Filtering": cf_final.mean()
})

display(final_comparison)

print("Baseline users:", len(baseline_final))
print("Collaborative-filtering users:", len(cf_final))

,Popularity Baseline,Item-Based Collaborative Filtering
Precision@10,0.058037,0.072758
Recall@10,0.049802,0.073998
NDCG@10,0.075741,0.090617


Baseline users: 591
Collaborative-filtering users: 591
